https://github.com/vllm-project/compressed-tensors

https://huggingface.co/models?search=W4A16

https://github.com/vllm-project/compressed-tensors/tree/main

https://docs.vllm.ai/en/latest/features/quantization/llm_compressor/int4/

https://aistudio.google.com/prompts/1hsBEt7FpMJk_SAsdHVoS0ZH-tFk52xz7

In [ ]:

الإجابة باختصار: لا، ليس دائماً، لكن صيغة W4A16 هي الأكثر شيوعاً واستخداماً بنسبة تزيد عن 90%.
أولاً: ماذا تعني الحروف والأرقام؟
في عالم تكميم النماذج (Quantization)، يتم تقسيم البيانات داخل النموذج إلى شقين:
W (Weights - الأوزان): هي أرقام النموذج الثابتة المحفوظة في ملفات الـ SafeTensors (التي قمت بضغطها إلى 4-bit لتقليل استهلاك الـ VRAM).
A (Activations - التنشيطات): هي الإشارات والأرقام التي تنتج لحظياً أثناء مرور السؤال والكلمات عبر طبقات النموذج أثناء التوليد.


خلاصة:
أي نموذج مكتوب عليه INT4 في الغالبية العظمى من الأدوات الشائعة (مثل vLLM, Ollama, GGUF/Q4, GPTQ, AWQ) هو فعلياً W4A16 (أوزان مضغوطة 4-bit لتقليل المساحة، وحسابات تجري بدقة 16-bit للحفاظ على العقل والذكاء).

In [1]:
%%bash
git clone https://github.com/vllm-project/compressed-tensors
cd compressed-tensors
pip install -e .

Obtaining file:///content/compressed-tensors
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.7 MB/s eta 0:00:00
  Building editable for compressed-tensors (pyproject.toml): started
  Building editable for compressed-tensors (pyproject.toml): finished with status 'done'
  Created wheel for compressed-tensors: filename=compressed_tensors-0.19.1.dev17+gb3966ea-0.editable-py3-none-any.whl size=9320 sha256=5cd9b4a25efdf686f422b4e0801a0fe6306e45217f114e1798b6c50e3fa37044
  Stored in directory: 

Cloning into 'compressed-tensors'...


In [4]:
from huggingface_hub import login

# ضع التوكن الخاص بك هنا
HF_TOKEN = "XXXXXXXXXXXXX"
login(token=HF_TOKEN)

In [1]:
cd compressed-tensors

/content/compressed-tensors


In [2]:
!python /content/compressed-tensors/examples/compress_w4a16.py

config.json: 100% 3.16k/3.16k [00:00<00:00, 2.05MB/s]
model.safetensors.index.json: 100% 76.2k/76.2k [00:00<00:00, 112MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/5.33G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/9.32G [00:00<?, ?B/s]
Reconstructing (incomplete total...):  43% 4.01G/9.32G [00:48<01:04, 82.3MB/s, 82.3MB/s  ]
Reconstructing (incomplete total...):  53% 4.96G/9.32G [01:18<01:48, 40.0MB/s, 40.0MB/s  ]
Reconstructing (incomplete total...):  58% 5.37G/9.32G [01:19<01:47, 36.9MB/s, 40.0MB/s  ]
Reconstructing (incomplete total...):  76% 7.12G/9.32G [01:30<00:33, 66.4MB/s, 72.4MB/s  ]
Reconstructing (incomplete total...):  87% 8.11G/9.32G [01:40<00:15, 78.9MB/s, 66.4MB/s  ]
Reconstructing (incomplete total...):  92% 8.62G/9.32G [01:41<00:06, 106MB/s, 78.9MB/s  ] 

Reconstructing (incomplete total...): 100% 9.32G/9.32G [

In [6]:
%cd /content

!git clone https://github.com/vllm-project/vllm.git

/content
Cloning into 'vllm'...
remote: Enumerating objects: 304663, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 304663 (delta 3), reused 2 (delta 2), pack-reused 304652 (from 2)
Receiving objects: 100% (304663/304663), 298.44 MiB | 18.65 MiB/s, done.
Resolving deltas: 100% (233455/233455), done.
Updating files: 100% (7579/7579), done.


In [ ]:
####
#
# The following example shows how a model can be quantized and packed
# entirely with primitives within `compressed-tensors`.
# The int4_config.json defines weights-only INT4 quantization,
# so we only need to compute weight scales (no calibration data needed).
# Weights are packed into int32 using the PackedQuantizationCompressor:
# src/compressed_tensors/compressors/pack_quantized/base.py
# Note: This is doing basic quantization. For better accuracy recovery,
# consider using GPTQ throught LLM-Compressor
#
####

# SPDX-License-Identifier: Apache-2.0
# SPDX-FileCopyrightText: Copyright contributors to the vLLM project

import math
from pathlib import Path

import torch
from compressed_tensors.compressors import ModelCompressor
from compressed_tensors.quantization import (
    QuantizationConfig,
    apply_quantization_config,
)
from compressed_tensors.quantization.utils import calculate_qparams
from transformers import AutoModelForCausalLM


model_name = "Qwen/Qwen3.5-4B"
device = "cuda:0" if torch.cuda.is_available() else "cpu"
# Load the model
model = AutoModelForCausalLM.from_pretrained(
    model_name, device_map=device, torch_dtype="auto"
)



# Set-up the quantization config from a JSON file. This defines:
# 1. What quantization scheme we're applying and to which layers
# 2. Any layers that should be ignored
# In this case, int4_config.json targets all Linear layers with weights-only INT4 quantization.
# All the quantization arguments are described through the QuantizationArgs,
# found in src/compressed_tensors/quantization/quant_args.py

config_file = Path(__file__).parent / "int4_config.json"
config = QuantizationConfig.model_validate_json(config_file.read_text())

# Apply the config to the model. This step uses the config to define
# the quantization parameters (such as the scales) for the targeted layers
# and attaches a QuantizationScheme which defines how the weights and activations
# should be quantized (e.g number of bits, group or block sizes, etc)
apply_quantization_config(model, config)

# Compute weight scales using round-to-nearest quantization
for name, module in model.named_modules():
    # Only target layers with a QuantizationScheme attached
    scheme = getattr(module, "quantization_scheme", None)
    if scheme is None or scheme.weights is None:
        continue

    weight = module.weight.data
    args = scheme.weights
    # Read the group_size defined in the user provided recipe
    # In this case, that is group_size=128 defined in int4_config.json
    group_size = args.group_size

    # Reshape based on the group_size. With group quantization, we apply a scale every
    # group_size number of rows. Reshape to make it easier to identify the min and max values per group.
    if group_size is not None and group_size > 0:
        reshaped = weight.unflatten(
            -1, (math.ceil(weight.shape[-1] / group_size), group_size)
        )
        min_val = reshaped.amin(dim=-1)
        max_val = reshaped.amax(dim=-1)
    else:
        min_val, max_val = torch.aminmax(weight)

    # Calculate the quantization parameters, such as the weight scale, using the min and max values
    scale, _ = calculate_qparams(min_val, max_val, args)
    # Update the parameters attached to the module based on the calculated value
    # In this case, we update the `weight_scale` attached to the targeted linear layers
    module.weight_scale.copy_(scale)

# set-up a compressor
# The compression format is inferred by iterating over all quantized modules and
# checking each format's `can_compress(module_type, scheme)` in priority order
# (see compressed_tensors/compressors/format.py). The format can also be set
# explicitly by passing quantization_format (e.g. quantization_format="pack-quantized")
compressor = ModelCompressor.from_pretrained_model(model)
# Compress the model using the calibrated scales and save it using the pack-quantized format.
# This format defines the weight packing, which can be seamlessly loaded through vLLM.
compressor.compress_model(model)

output_dir = "./Qwen/Qwen3.5-4B-W4A16"
# With pack-quantized, weights are quantized and then packed into int32 tensors
# (e.g. eight 4-bit values per int32). The saved state dict stores weight_packed,
# weight_scale, and weight_shape (replacing the original weight tensor).
model.save_pretrained(output_dir)
# Update the model's config with the relevant compressed-tensors details
compressor.update_config(output_dir)


In [ ]:
# SPDX-License-Identifier: Apache-2.0
# SPDX-FileCopyrightText: Copyright contributors to the vLLM project

from vllm import LLM, SamplingParams

# Sample prompts.
prompts = [
    "Hello, my name is",
    "The president of the United States is",
    "The capital of France is",
    "The future of AI is",
]
# Create a sampling params object.
sampling_params = SamplingParams(temperature=0.8, top_p=0.95)


def main():
    # Create an LLM.
    llm = LLM(model="facebook/opt-125m")
    # Generate texts from the prompts.
    # The output is a list of RequestOutput objects
    # that contain the prompt, generated text, and other information.
    outputs = llm.generate(prompts, sampling_params)
    # Print the outputs.
    print("\nGenerated Outputs:\n" + "-" * 60)
    for output in outputs:
        prompt = output.prompt
        generated_text = output.outputs[0].text
        print(f"Prompt:    {prompt!r}")
        print(f"Output:    {generated_text!r}")
        print("-" * 60)


if __name__ == "__main__":
    main()


In [ ]:
%cd /content/vllm/examples/basic/offline_inference
!python /content/vllm/examples/basic/offline_inference/basic.py

In [ ]:
/content/compressed-tensors/Qwen/Qwen3.5-4B-W4A16

In [13]:
!rm -rf /content/compressed-tensors/Qwen/Qwen3.5-4B-W4A16

In [14]:
import math
import shutil
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM
from compressed_tensors.quantization import QuantizationConfig, apply_quantization_config
from compressed_tensors.quantization.utils import calculate_qparams
from compressed_tensors.compressors import ModelCompressor

# 1. Clean up old checkpoint
output_dir = "./Qwen/Qwen3.5-4B-W4A16"
shutil.rmtree(output_dir, ignore_errors=True)

model_name = "Qwen/Qwen3.5-4B"
device = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f"Loading {model_name}...")
model = AutoModelForCausalLM.from_pretrained(
    model_name, device_map=device, torch_dtype="auto"
)

# 2. Load quantization config and explicitly IGNORE lm_head
config_file = Path("/content/compressed-tensors/examples/int4_config.json")
config = QuantizationConfig.model_validate_json(config_file.read_text())
config.ignore = ["lm_head"]  # <-- CRITICAL: keeps lm_head in unquantized FP16/BF16

apply_quantization_config(model, config)

# 3. Compute weight scales (lm_head is skipped automatically)
print("Computing weight scales...")
for name, module in model.named_modules():
    scheme = getattr(module, "quantization_scheme", None)
    if scheme is None or scheme.weights is None:
        continue

    weight = module.weight.data
    args = scheme.weights
    group_size = args.group_size

    if group_size is not None and group_size > 0:
        reshaped = weight.unflatten(
            -1, (math.ceil(weight.shape[-1] / group_size), group_size)
        )
        min_val = reshaped.amin(dim=-1)
        max_val = reshaped.amax(dim=-1)
    else:
        min_val, max_val = torch.aminmax(weight)

    scale, _ = calculate_qparams(min_val, max_val, args)
    module.weight_scale.copy_(scale)

# 4. Pack and save
print("Packing and saving compressed model...")
compressor = ModelCompressor.from_pretrained_model(model)
compressor.compress_model(model)

model.save_pretrained(output_dir)
compressor.update_config(output_dir)
print(f"Done! Saved to {output_dir}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [21]:
!rm -rf /content/vllm/examples/basic/offline_inference/Qwen/Qwen3.5-4B-W4A16

In [ ]:
%cd /content/vllm/examples/basic/offline_inference

In [22]:
import math
import shutil
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM
from compressed_tensors.quantization import QuantizationConfig, apply_quantization_config
from compressed_tensors.quantization.utils import calculate_qparams
from compressed_tensors.compressors import ModelCompressor

# مسار الحفظ النهائي الموحد
output_dir = "/content/Qwen3.5-4B-W4A16"
shutil.rmtree(output_dir, ignore_errors=True)

model_name = "Qwen/Qwen3.5-4B"
device = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f"1. Loading base model: {model_name}...")
model = AutoModelForCausalLM.from_pretrained(
    model_name, device_map=device, torch_dtype="auto"
)

# قراءة الإعدادات واستثناء lm_head
config_file = Path("/content/compressed-tensors/examples/int4_config.json")
config = QuantizationConfig.model_validate_json(config_file.read_text())
config.ignore = ["lm_head"]  # استثناء lm_head ليقبلها vLLM

apply_quantization_config(model, config)

print("2. Calculating quantization scales...")
for name, module in model.named_modules():
    scheme = getattr(module, "quantization_scheme", None)
    if scheme is None or scheme.weights is None:
        continue

    weight = module.weight.data
    group_size = scheme.weights.group_size

    if group_size is not None and group_size > 0:
        reshaped = weight.unflatten(
            -1, (math.ceil(weight.shape[-1] / group_size), group_size)
        )
        min_val = reshaped.amin(dim=-1)
        max_val = reshaped.amax(dim=-1)
    else:
        min_val, max_val = torch.aminmax(weight)

    scale, _ = calculate_qparams(min_val, max_val, scheme.weights)
    module.weight_scale.copy_(scale)

print("3. Compressing and saving model...")
compressor = ModelCompressor.from_pretrained_model(model)
compressor.compress_model(model)

model.save_pretrained(output_dir)
compressor.update_config(output_dir)
print(f" Compression complete! Saved to {output_dir}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
%%writefile run_infer.py
from vllm import LLM, SamplingParams

prompts = [
    "Hello, my name is",
    "The president of the United States is",
    "The capital of France is",
    "The future of AI is",
]

sampling_params = SamplingParams(temperature=0.8, top_p=0.95, max_tokens=100)

def main():
    llm = LLM(
        model="/content/Qwen3.5-4B-W4A16",
        max_model_len=4096,
        gpu_memory_utilization=0.85,
        enforce_eager=True,  # لتقليل استهلاك الذاكرة وتفادي مشاكل Inductor/Graph على T4
    )

    outputs = llm.generate(prompts, sampling_params)
    print("\n" + "=" * 50)
    for output in outputs:
        print(f"\nPrompt: {output.prompt}")
        print(f"Output: {output.outputs[0].text}")
    print("=" * 50)

if __name__ == "__main__":
    main()

In [24]:
from safetensors.torch import load_file

# قراءة ملف الأوزان
weights = load_file("/content/Qwen3.5-4B-W4A16/model.safetensors")

# فحص طبقة عادية (يجب أن تظهر مضغوطة):
print("Backbone layer weight key:", [k for k in weights.keys() if "q_proj" in k][:3])
# ستلاحظ وجود: q_proj.weight_packed, q_proj.weight_scale, q_proj.weight_shape

# فحص طبقة المخرجات (يجب أن تظل unquantized):
assert "lm_head.weight" in weights, "خطأ: lm_head تم ضغطها بالخطأ!"
print("lm_head shape and dtype:", weights["lm_head.weight"].shape, weights["lm_head.weight"].dtype)
print(" تم التحقق الهيكلي بنجاح!")

AssertionError: خطأ: lm_head تم ضغطها بالخطأ!

In [2]:
!pwd

/content


In [ ]:
pip install lm-eval

In [5]:
%%writefile run_infer.py
from vllm import LLM, SamplingParams

prompts = [
    "Hello, my name is",
    "The capital of France is",
    "The future of artificial intelligence is",
]

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=60,
)

def main():
    print("Loading quantized model with vLLM...")
    llm = LLM(
        model="/content/Qwen3.5-4B-W4A16",
        max_model_len=2048,
        gpu_memory_utilization=0.85,
        enforce_eager=True,
    )

    outputs = llm.generate(prompts, sampling_params)
    print("\n" + "=" * 50)
    for output in outputs:
        print(f"\nPrompt: {output.prompt}")
        print(f"Output: {output.outputs[0].text.strip()}")
    print("\n" + "=" * 50)

if __name__ == "__main__":
    main()

Writing run_infer.py


In [1]:
!python run_infer.py

Loading quantized model with vLLM...
INFO 10-02 22:06:20 [api_utils.py:286] non-default args: {'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'model': '/content/Qwen3.5-4B-W4A16'}
INFO 10-02 22:06:20 [model.py:692] Resolved architecture: Qwen3_5ForCausalLM
WARNING 10-02 22:06:20 [model.py:2311] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 10-02 22:06:20 [model.py:2364] Casting torch.bfloat16 to torch.float16.
INFO 10-02 22:06:20 [model.py:2030] Using max model len 2048
INFO 10-02 22:06:22 [scheduler.py:288] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 10-02 22:06:22 [config.py:625] Mamba cache mode is set to 'align' for Qwen3_5ForCausalLM by default when prefix caching is enabled
WARNING 10-02 22:06:24 [vllm.py:1547] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
print("تم تفريغ ذاكرة كارت الشاشة بنجاح!")

In [ ]:
%%writefile infer.py
import os
# إجبار vLLM على العمل بذاكرة نظيفة
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

from vllm import LLM, SamplingParams

prompts = [
    "Hello, my name is",
    "The capital of France is",
    "What is the future of AI?",
]

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=50,
)

def main():
    # تقليل النسبة إلى 0.75 لتترك مساحة كافية على كارت T4
    llm = LLM(
        model="/content/Qwen3.5-4B-W4A16",
        max_model_len=2048,
        gpu_memory_utilization=0.75,
        enforce_eager=True,
    )

    outputs = llm.generate(prompts, sampling_params)
    print("\n" + "=" * 50)
    for output in outputs:
        print(f"\nPrompt: {output.prompt}")
        print(f"Output: {output.outputs[0].text.strip()}")
    print("\n" + "=" * 50)

if __name__ == "__main__":
    main()

In [ ]:
!python infer.py

### aشغال

In [3]:
%%writefile /content/1.py
import os
from vllm import LLM, SamplingParams

prompts = [
    "Hello, my name is",
    "The capital of France is",
    "What is the future of AI?",
]

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=60,
)

def main():
    print(">>> Initializing vLLM Engine...")
    llm = LLM(
        model="/content/Qwen3.5-4B-W4A16",
        tokenizer="Qwen/Qwen3.5-4B",  # <-- إضافة هذا السطر لحل المشكلة فوراً
        max_model_len=2048,
        gpu_memory_utilization=0.75,
        enforce_eager=True,
    )

    print(">>> Generating outputs...")
    outputs = llm.generate(prompts, sampling_params)

    print("\n" + "=" * 50)
    for output in outputs:
        print(f"\nPrompt: {output.prompt}")
        print(f"Output: {output.outputs[0].text.strip()}")
    print("\n" + "=" * 50)

if __name__ == "__main__":
    main()

Overwriting /content/1.py


In [4]:
!python /content/1.py

>>> Initializing vLLM Engine...
INFO 10-02 22:20:59 [api_utils.py:286] non-default args: {'tokenizer': 'Qwen/Qwen3.5-4B', 'max_model_len': 2048, 'gpu_memory_utilization': 0.75, 'disable_log_stats': True, 'enforce_eager': True, 'model': '/content/Qwen3.5-4B-W4A16'}
INFO 10-02 22:21:00 [model.py:692] Resolved architecture: Qwen3_5ForCausalLM
WARNING 10-02 22:21:00 [model.py:2311] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 10-02 22:21:00 [model.py:2364] Casting torch.bfloat16 to torch.float16.
INFO 10-02 22:21:00 [model.py:2030] Using max model len 2048
INFO 10-02 22:21:02 [scheduler.py:288] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 10-02 22:21:02 [config.py:625] Mamba cache mode is set to 'align' for Qwen3_5ForCausalLM by default when prefix caching is enabled
WARNING 10-02 22:21:02 [vllm.py:1547] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equ

In [8]:
%%writefile /content/1.py
import os
from vllm import LLM, SamplingParams

prompts = [
    "Who was Napoleon Bonaparte?",
]

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=128,
)

def main():
    print(">>> Initializing vLLM Engine...")
    llm = LLM(
        model="/content/Qwen3.5-4B-W4A16",
        tokenizer="Qwen/Qwen3.5-4B",  # <-- إضافة هذا السطر لحل المشكلة فوراً
        max_model_len=2048,
        gpu_memory_utilization=0.75,
        enforce_eager=True,
    )

    print(">>> Generating outputs...")
    outputs = llm.generate(prompts, sampling_params)

    print("\n" + "=" * 50)
    for output in outputs:
        print(f"\nPrompt: {output.prompt}")
        print(f"Output: {output.outputs[0].text.strip()}")
    print("\n" + "=" * 50)

if __name__ == "__main__":
    main()

Overwriting /content/1.py


In [9]:
!python /content/1.py

>>> Initializing vLLM Engine...
INFO 10-02 22:30:22 [api_utils.py:286] non-default args: {'tokenizer': 'Qwen/Qwen3.5-4B', 'max_model_len': 2048, 'gpu_memory_utilization': 0.75, 'disable_log_stats': True, 'enforce_eager': True, 'model': '/content/Qwen3.5-4B-W4A16'}
INFO 10-02 22:30:22 [model.py:692] Resolved architecture: Qwen3_5ForCausalLM
WARNING 10-02 22:30:22 [model.py:2311] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 10-02 22:30:22 [model.py:2364] Casting torch.bfloat16 to torch.float16.
INFO 10-02 22:30:22 [model.py:2030] Using max model len 2048
INFO 10-02 22:30:24 [scheduler.py:288] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 10-02 22:30:24 [config.py:625] Mamba cache mode is set to 'align' for Qwen3_5ForCausalLM by default when prefix caching is enabled
WARNING 10-02 22:30:24 [vllm.py:1547] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equ

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. تحميل الـ Tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B")

# 2. تحميل النموذج مع تحديد float16 لتجنب حجز مساحات إضافية
model = AutoModelForCausalLM.from_pretrained(
    "/content/Qwen3.5-4B-W4A16",
    device_map="auto",
    dtype=torch.float16,
)

# 3. إعداد قالب المحادثة
messages = [
    {"role": "user", "content": "Who are you?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

# 4. التوليد
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        temperature=0.7,
        do_sample=True,
    )

# 5. طباعة الرد فقط
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print("\nModel Response:\n" + "-" * 30)
print(response)

Compressing model: 100%|██████████| 248/248 [00:02<00:00, 94.60it/s]


Loading weights:   0%|          | 0/922 [00:00<?, ?it/s]

Decompressing model: 100%|██████████| 248/248 [00:03<00:00, 79.51it/s]
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.



Model Response:
------------------------------
Okay, the user asked "Who are you?" That's a straightforward question. I need to respond as Qwen3.5, the latest version of Qwen from Alibaba Cloud. Let me make


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. تحميل الـ Tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B")

# 2. تحميل النموذج مع تحديد float16 لتجنب حجز مساحات إضافية
model = AutoModelForCausalLM.from_pretrained(
    "/content/Qwen3.5-4B-W4A16",
    device_map="auto",
    dtype=torch.float16,
)

# 3. إعداد قالب المحادثة
messages = [
    {"role": "user", "content": "what is ai?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

# 4. التوليد
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        temperature=0.7,
        do_sample=True,
    )

# 5. طباعة الرد فقط
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print("\nModel Response:\n" + "-" * 30)
print(response)

Compressing model: 100%|██████████| 248/248 [00:02<00:00, 118.63it/s]


Loading weights:   0%|          | 0/922 [00:00<?, ?it/s]

Decompressing model: 100%|██████████| 248/248 [00:02<00:00, 113.23it/s]
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.



Model Response:
------------------------------
Okay, the user is asking "what is ai?" Let me start by recalling the definition of AI. Artificial Intelligence is a branch of computer science focused on creating systems that can perform tasks that typically require


In [ ]:
AtlaAI/Selene-1-Mini-Llama-3.1-8B-GPTQ-W4A16

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. تحميل الـ Tokenizer
tokenizer = AutoTokenizer.from_pretrained("AtlaAI/Selene-1-Mini-Llama-3.1-8B-GPTQ-W4A16")

# 2. تحميل النموذج مع تحديد float16 لتجنب حجز مساحات إضافية
model = AutoModelForCausalLM.from_pretrained(
    "AtlaAI/Selene-1-Mini-Llama-3.1-8B-GPTQ-W4A16",
    device_map="auto",
    dtype=torch.float16,
)

# 3. إعداد قالب المحادثة
messages = [
    {"role": "user", "content": "what is ai?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

# 4. التوليد
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        temperature=0.7,
        do_sample=True,
    )

# 5. طباعة الرد فقط
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print("\nModel Response:\n" + "-" * 30)
print(response)

Compressing model: 100%|██████████| 224/224 [00:03<00:00, 70.20it/s]


Loading weights:   0%|          | 0/739 [00:00<?, ?it/s]

Decompressing model:  86%|████████▌ | 193/224 [00:03<00:00, 63.08it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 224.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 161.75 MiB is free. Including non-PyTorch memory, this process has 14.40 GiB memory in use. Of the allocated memory 13.81 GiB is allocated by PyTorch, and 485.38 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("AtlaAI/Selene-1-Mini-Llama-3.1-8B-GPTQ-W4A16")
model = AutoModelForCausalLM.from_pretrained("AtlaAI/Selene-1-Mini-Llama-3.1-8B-GPTQ-W4A16", device_map="auto")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

In [ ]:
import os
os._exit(0)

In [1]:
%%writefile run_8b.py
from vllm import LLM, SamplingParams

prompts = [
    "What is AI?",
    "Explain quantum computing in simple terms.",
]

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=100,
)

def main():
    print(">>> Loading Llama-3.1-8B-GPTQ-W4A16 into vLLM...")
    llm = LLM(
        model="AtlaAI/Selene-1-Mini-Llama-3.1-8B-GPTQ-W4A16",
        max_model_len=2048,
        gpu_memory_utilization=0.85, # سيأخذ حوالي 5.5 جيجابايت فقط للأوزان
        enforce_eager=True,
    )

    outputs = llm.generate(prompts, sampling_params)
    print("\n" + "=" * 50)
    for output in outputs:
        print(f"\nPrompt: {output.prompt}")
        print(f"Output: {output.outputs[0].text.strip()}")
    print("\n" + "=" * 50)

if __name__ == "__main__":
    main()

Writing run_8b.py


In [2]:
!python run_8b.py

>>> Loading Llama-3.1-8B-GPTQ-W4A16 into vLLM...
INFO 10-02 22:49:02 [api_utils.py:286] non-default args: {'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'AtlaAI/Selene-1-Mini-Llama-3.1-8B-GPTQ-W4A16'}
INFO 10-02 22:49:23 [model.py:692] Resolved architecture: LlamaForCausalLM
WARNING 10-02 22:49:23 [model.py:2311] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 10-02 22:49:23 [model.py:2364] Casting torch.bfloat16 to torch.float16.
INFO 10-02 22:49:23 [model.py:2030] Using max model len 2048
INFO 10-02 22:49:24 [scheduler.py:288] Chunked prefill is enabled with max_num_batched_tokens=8192.
Traceback (most recent call last):
  File "/content/run_8b.py", line 31, in <module>
    main()
    ~~~~^^
  File "/content/run_8b.py", line 16, in main
    llm = LLM(
        model="AtlaAI/Selene-1-Mini-Llama-3.1-8B-GPTQ-W4A16",
    ...<2 li

In [ ]:
max_memory = {0: "12GiB", "cpu": "20GiB"}

model = AutoModelForCausalLM.from_pretrained(
    "AtlaAI/Selene-1-Mini-Llama-3.1-8B-GPTQ-W4A16",
    device_map="auto",
    max_memory=max_memory,
    dtype=torch.float16,
)

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. تحميل الـ Tokenizer
tokenizer = AutoTokenizer.from_pretrained("AtlaAI/Selene-1-Mini-Llama-3.1-8B-GPTQ-W4A16")

max_memory = {0: "12GiB", "cpu": "20GiB"}

model = AutoModelForCausalLM.from_pretrained(
    "AtlaAI/Selene-1-Mini-Llama-3.1-8B-GPTQ-W4A16",
    device_map="auto",
    max_memory=max_memory,
    dtype=torch.float16,
)

# 3. إعداد قالب المحادثة
messages = [
    {"role": "user", "content": "what is ai?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

# 4. التوليد
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        temperature=0.7,
        do_sample=True,
    )

# 5. طباعة الرد فقط
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print("\nModel Response:\n" + "-" * 30)
print(response)

Compressing model: 100%|██████████| 224/224 [00:04<00:00, 51.16it/s]


Loading weights:   0%|          | 0/739 [00:00<?, ?it/s]

Decompressing model:  83%|████████▎ | 187/224 [00:03<00:00, 47.11it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 224.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 159.81 MiB is free. Including non-PyTorch memory, this process has 14.40 GiB memory in use. Of the allocated memory 13.59 GiB is allocated by PyTorch, and 715.38 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [1]:
%%writefile run_selene.py
import json
from pathlib import Path
from huggingface_hub import snapshot_download
from vllm import LLM, SamplingParams

# الحصول على مسار النموذج المحلي المعدل
model_dir = snapshot_download(repo_id="AtlaAI/Selene-1-Mini-Llama-3.1-8B-GPTQ-W4A16")

prompts = [
    "What is AI?",
    "Explain the concept of quantum computing simply.",
]

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=80,
)

def main():
    print(">>> Loading patched Llama-3.1-8B into vLLM...")
    llm = LLM(
        model=model_dir,
        max_model_len=2048,
        gpu_memory_utilization=0.75, # 75% كافية جداً لـ 8B INT4 على T4
        enforce_eager=True,
    )

    outputs = llm.generate(prompts, sampling_params)
    print("\n" + "=" * 50)
    for output in outputs:
        print(f"\nPrompt: {output.prompt}")
        print(f"Output: {output.outputs[0].text.strip()}")
    print("\n" + "=" * 50)

if __name__ == "__main__":
    main()

Writing run_selene.py


In [2]:
!python run_selene.py

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0% 0/11 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/1.57k [00:00<?, ?B/s]         

Fetching 11 files:  82% 9/11 [00:00<00:00, 22.93it/s]
Reconstructing (incomplete total...):  91% 1.57k/1.72k [00:00<00:00, 4.01kB/s]
Fetching 11 files: 100% 11/11 [00:00<00:00, 24.98it/s]
Download complete: 100% 4.92k/4.92k [00:00<00:00, 4.01kB/s]
Reconstruction complete: 100% 4.92k/4.92k [00:00<00:00, 4.01kB/s]             >>> Loading patched Llama-3.1-8B into vLLM...
INFO 10-02 22:53:47 [api_utils.py:286] non-default args: {'max_model_len': 2048, 'gpu_memory_utilization': 0.75, 'disable_log_stats': True, 'enforce_eager': True, 'model': '/root/.cache/huggingface/hub/models--AtlaAI--Selene-1-Mini-Llama-3.1-8B-GPTQ-W4A16/snapshots/0405eef72fe7144f210090ea60d149407a24921e'}
INFO 10-02 22:53:47 [model.py:692] Resolved architecture: LlamaForCausalLM
WARNING 10-02 22:53:47 [model.py:2311]

### aشغال

In [3]:
%%writefile run_selene_fixed.py
import os
import json
import shutil
from pathlib import Path
from huggingface_hub import snapshot_download
from vllm import LLM, SamplingParams

# 1. تنزيل النموذج إلى مجلد محلي ثابت خارج كاش الـ Hub
model_source = "AtlaAI/Selene-1-Mini-Llama-3.1-8B-GPTQ-W4A16"
target_dir = Path("/content/Selene-8B-Patched")

if not target_dir.exists():
    print(">>> 1. Downloading and copying model to independent local directory...")
    cached_path = snapshot_download(repo_id=model_source)
    # نسخ المجلد بالكامل مع فك روابط الـ symlinks ليصبح ملفات حقيقية مستقلة
    shutil.copytree(cached_path, target_dir, symlinks=False)

    # 2. تعديل config.json وحذف sparsity_config نهائياً
    config_file = target_dir / "config.json"
    config_data = json.loads(config_file.read_text())

    if "quantization_config" in config_data:
        # إزالة أي ذكر للـ sparsity
        config_data["quantization_config"].pop("sparsity_config", None)
        config_data["quantization_config"].pop("sparsity", None)

    config_file.write_text(json.dumps(config_data, indent=2))
    print(">>> Successfully patched config.json in local directory!")
else:
    print(">>> Using existing patched model directory:", target_dir)

# 3. إعداد مدخلات الاختبار
prompts = [
    "What is AI?",
    "Explain the concept of quantum computing simply.",
]

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=80,
)

def main():
    print(">>> Loading local patched Llama-3.1-8B into vLLM...")
    llm = LLM(
        model=str(target_dir),  # تمرير المجلد المحلي المعدل
        max_model_len=2048,
        gpu_memory_utilization=0.75,
        enforce_eager=True,
    )

    outputs = llm.generate(prompts, sampling_params)
    print("\n" + "=" * 50)
    for output in outputs:
        print(f"\nPrompt: {output.prompt}")
        print(f"Output: {output.outputs[0].text.strip()}")
    print("\n" + "=" * 50)

if __name__ == "__main__":
    main()

Writing run_selene_fixed.py


In [4]:
!python run_selene_fixed.py

>>> 1. Downloading and copying model to independent local directory...
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files: 100% 11/11 [00:00<00:00, 1224.26it/s]
Download complete: :           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B                         >>> Successfully patched config.json in local directory!
>>> Loading local patched Llama-3.1-8B into vLLM...
Download complete: :           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            
INFO 10-02 22:56:17 [api_utils.py:286] non-default args: {'max_model_len': 2048, 'gpu_memory_utilization': 0.75, 'disable_log_stats': True, 'enforce_eager': True, 'model': '/content/Selene-8B-Patched'}
INFO 10-02 22:56:17 [model.py:692] Resolved architecture: LlamaForCausalLM
WARNING 10-02 22:56:17 [model.py:2311] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for 